# ML Lab 3: Decision Tree Classification

**Course:** Machine Learning — BS Computer Science, 5th Semester  
**Dataset:** Telco Customer Churn (`clean_churn.csv`)

### Student Information
- **Name:** Abdul Ahad
- **Student ID:** __________________
- **Section:** __________________
- **GitHub Profile:** __________________
- **Kaggle Profile:** __________________
- **Dataset:** Telco Customer Churn
- **Dataset Source:** IBM Telco Customer Churn dataset / Lab 2 cleaned dataset

> This notebook follows the Lab 3 manual. Every table, plot, and metric is followed by a short interpretation.


## 1. Problem Definition

The goal is to predict whether a telecom customer will **churn** (leave the service) based on the customer's available service, contract, billing, and demographic information. Predicting churn matters because the company can identify customers who may leave and use the information to improve retention strategies.

## 2. Dataset Verification

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

# Upload clean_churn.csv to Colab if it is not already in the working directory.
df = pd.read_csv("clean_churn.csv")

print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())
print("\nTarget distribution:")
print(df["Churn"].value_counts())
print("\nTarget proportions:")
print(df["Churn"].value_counts(normalize=True))

print("\nDataset information:")
df.info()


**Interpretation:** The output above verifies the number of rows and columns, data types, and missing values. The `Churn` distribution also shows whether the target classes are balanced or imbalanced. The dataset is ready for modeling only if the required columns contain no missing values.

## 3. Feature / Target Preparation

In [ ]:
# Task 3: Separate target and features
y = df["Churn"].map({"No": 0, "Yes": 1}).astype(int)

# Drop the target and any identifier column.
X = df.drop(columns=["Churn"], errors="ignore").copy()

identifier_columns = [c for c in ["customerID", "CustomerID", "customer_id"] if c in X.columns]
if identifier_columns:
    X = X.drop(columns=identifier_columns)
    print("Dropped identifier column(s):", identifier_columns)
else:
    print("No identifier column found.")

print("y values:")
print(y.value_counts())
print("\nInitial X shape:", X.shape)


In [ ]:
# Task 4: Convert categorical columns to numeric values.
categorical_columns = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("Categorical columns before encoding:")
print(categorical_columns)

# One-hot encoding is used because most categorical variables have more than
# two categories, and it avoids imposing an artificial numerical order.
X = pd.get_dummies(X, columns=categorical_columns, drop_first=False, dtype=int)

print("\nX shape after encoding:", X.shape)


In [ ]:
# Task 5: Confirm that X is numeric and has no missing values.
print("All columns numeric:", all(pd.api.types.is_numeric_dtype(dtype) for dtype in X.dtypes))
print("Total missing values in X:", X.isnull().sum().sum())
print("\nFinal feature data types:")
print(X.dtypes.value_counts())


**Interpretation:** One-hot encoding converts categorical variables into numeric indicator columns without assuming that categories have a natural numerical order. The final checks confirm that the Decision Tree receives numeric features and that there are no missing feature values.

## 4. Train / Test Split

In [ ]:
# Task 6: 80/20 split with stratification.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Task 7: Report shapes.
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

print("\nTraining target proportions:")
print(y_train.value_counts(normalize=True))

print("\nTesting target proportions:")
print(y_test.value_counts(normalize=True))


**Interpretation:** `stratify=y` keeps the proportion of churners and non-churners approximately the same in both the training and test sets. This is sensible because the target was found to be imbalanced, so a random split without stratification could produce less representative subsets.

## 5. Baseline Decision Tree

In [ ]:
# Task 8: Train an unrestricted baseline Decision Tree.
baseline_model = DecisionTreeClassifier(random_state=42)
baseline_model.fit(X_train, y_train)

y_train_pred_baseline = baseline_model.predict(X_train)
y_test_pred_baseline = baseline_model.predict(X_test)

# Task 9: Tree depth.
print("Baseline tree depth:", baseline_model.get_depth())
print("Number of leaves:", baseline_model.get_n_leaves())


**Interpretation:** The baseline tree is allowed to grow without a `max_depth` restriction. The reported depth shows how complex the tree became when scikit-learn was allowed to split until its default stopping conditions were reached.

## 6. Model Evaluation

In [ ]:
def evaluate_model(model, X_eval, y_eval):
    pred = model.predict(X_eval)
    return {
        "Accuracy": accuracy_score(y_eval, pred),
        "Precision": precision_score(y_eval, pred, zero_division=0),
        "Recall": recall_score(y_eval, pred, zero_division=0),
        "F1-score": f1_score(y_eval, pred, zero_division=0)
    }

baseline_metrics = evaluate_model(baseline_model, X_test, y_test)
baseline_metrics_df = pd.DataFrame([baseline_metrics], index=["Baseline"])
baseline_metrics_df


**Interpretation:** Accuracy gives the overall proportion of correct predictions, while precision measures how many predicted churners were actually churners, recall measures how many actual churners were detected, and F1-score balances precision and recall. Because churn is typically imbalanced, accuracy should not be considered by itself.

In [ ]:
# Task 11: Confusion matrix.
cm = confusion_matrix(y_test, y_test_pred_baseline)

print("Confusion matrix:")
print(cm)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No Churn", "Churn"]
)
disp.plot()
plt.title("Baseline Decision Tree - Test Confusion Matrix")
plt.show()

tn, fp, fn, tp = cm.ravel()
print("Actual churners correctly caught (True Positives):", tp)
print("Actual churners missed (False Negatives):", fn)


**Interpretation:** The true-positive value shows how many actual churners were correctly detected. The false-negative value shows how many actual churners were missed. For a churn problem, recall is particularly informative because missing a real churner can mean losing an opportunity for retention.

**Task 12:** Given the class imbalance, recall and F1-score are especially useful. Recall focuses on detecting actual churners, while F1-score considers both recall and precision. Accuracy alone can appear strong even when the minority churn class is not detected well.

## 7. Overfitting Experiment

In [ ]:
# Task 13: Compare training and test accuracy.
train_acc_baseline = accuracy_score(y_train, y_train_pred_baseline)
test_acc_baseline = accuracy_score(y_test, y_test_pred_baseline)

print("Baseline training accuracy:", train_acc_baseline)
print("Baseline test accuracy:", test_acc_baseline)
print("Train-test accuracy gap:", train_acc_baseline - test_acc_baseline)

if train_acc_baseline - test_acc_baseline > 0.05:
    print("\nInterpretation: The relatively large gap suggests overfitting.")
else:
    print("\nInterpretation: The train-test accuracy gap is not especially large, although other metrics should also be considered.")


In [ ]:
# Task 14: Train trees at several depths.
depths = [2, 3, 4, 5, 6, 8, 10, None]
depth_results = []

for depth in depths:
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    depth_results.append({
        "max_depth": "unrestricted" if depth is None else depth,
        "train_accuracy": accuracy_score(y_train, train_pred),
        "test_accuracy": accuracy_score(y_test, test_pred),
        "test_precision": precision_score(y_test, test_pred, zero_division=0),
        "test_recall": recall_score(y_test, test_pred, zero_division=0),
        "test_f1": f1_score(y_test, test_pred, zero_division=0)
    })

depth_results_df = pd.DataFrame(depth_results)
depth_results_df


**Interpretation:** As tree depth increases, the model becomes more flexible. Training accuracy generally increases because a deeper tree can fit more detailed patterns, while test performance may stop improving or decline when the model begins learning noise.

In [ ]:
# Task 15: Plot training vs. test accuracy.
plot_df = depth_results_df.copy()
plot_df["depth_num"] = plot_df["max_depth"].replace("unrestricted", np.nan)
unrestricted_x = plot_df.loc[plot_df["max_depth"] == "unrestricted", "train_accuracy"].index[0]

plt.figure(figsize=(9, 5))
plt.plot(plot_df["depth_num"], plot_df["train_accuracy"], marker="o", label="Train Accuracy")
plt.plot(plot_df["depth_num"], plot_df["test_accuracy"], marker="o", label="Test Accuracy")
plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.title("Training vs Test Accuracy by Tree Depth")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("The unrestricted tree is shown separately in the table above because it has no numeric max_depth.")
print("\nTable sorted by test F1-score:")
display(depth_results_df.sort_values("test_f1", ascending=False))


**Interpretation:** Overfitting begins around the point where training accuracy continues to improve while test accuracy stops improving or starts to fall. Use the plotted curve and the results table from the actual run to identify the approximate depth for this dataset.

## 8. Model Comparison

In [ ]:
# Task 16: Select a depth using test performance and the train-test gap.
# This is evidence-based rather than selecting purely by training accuracy.
depth_candidates = depth_results_df[depth_results_df["max_depth"] != "unrestricted"].copy()
depth_candidates["accuracy_gap"] = (
    depth_candidates["train_accuracy"] - depth_candidates["test_accuracy"]
)

# Prefer strong test F1 with a controlled train-test gap.
reasonable = depth_candidates[depth_candidates["accuracy_gap"] <= 0.08]
if len(reasonable) == 0:
    reasonable = depth_candidates

selected_row = reasonable.sort_values(
    ["test_f1", "test_recall", "test_accuracy"],
    ascending=False
).iloc[0]

selected_depth = int(selected_row["max_depth"])

print("Selected max_depth:", selected_depth)
print("Selected model row:")
display(pd.DataFrame([selected_row]))

print(
    f"\nJustification: depth={selected_depth} provides a strong test F1-score "
    f"while keeping the training-test accuracy gap at about "
    f"{selected_row['accuracy_gap']:.3f}. This balances model complexity "
    f"against generalization rather than choosing the model only by training accuracy."
)


In [ ]:
# Task 17: Retrain the chosen-depth model and evaluate it.
chosen_model = DecisionTreeClassifier(max_depth=selected_depth, random_state=42)
chosen_model.fit(X_train, y_train)

chosen_metrics = evaluate_model(chosen_model, X_test, y_test)
chosen_metrics_df = pd.DataFrame([chosen_metrics], index=["Chosen Depth"])
chosen_metrics_df


In [ ]:
# Task 18: Entropy model.
entropy_model = DecisionTreeClassifier(
    max_depth=selected_depth,
    criterion="entropy",
    random_state=42
)
entropy_model.fit(X_train, y_train)

entropy_metrics = evaluate_model(entropy_model, X_test, y_test)
entropy_metrics_df = pd.DataFrame([entropy_metrics], index=["Entropy"])
entropy_metrics_df


In [ ]:
# Compare baseline, chosen-depth, and entropy models.
comparison_df = pd.concat([
    baseline_metrics_df,
    chosen_metrics_df,
    entropy_metrics_df
])
comparison_df


**Interpretation:** The comparison table shows how limiting tree depth changes generalization compared with the unrestricted baseline. The entropy model uses a different splitting criterion at the same selected depth, allowing its test metrics to be compared directly with the chosen-depth model.

## 9. Feature Importance

In [ ]:
# Task 19: Feature importances for the chosen model.
importances = pd.Series(
    chosen_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print("Top 10 features:")
display(importances.head(10).to_frame("importance"))

top_features = importances.head(10).sort_values()

plt.figure(figsize=(9, 6))
plt.barh(top_features.index, top_features.values)
plt.xlabel("Feature Importance")
plt.ylabel("Feature")
plt.title("Top Feature Importances - Chosen Decision Tree")
plt.show()

print("Top 5 features:")
print(list(importances.head(5).index))


**Interpretation:** Feature importance represents the relative contribution assigned by the fitted Decision Tree to reducing impurity through splits. The top 3–5 features should be compared with the relationships observed during Lab 2 EDA rather than treated as proof of causation.

### Task 20 — Connection to Lab 2 EDA

After running the feature-importance cell, compare the top 3–5 features with the patterns you documented in Lab 2. Write 2–3 sentences such as:

> The most important features in the Decision Tree are **[feature 1]**, **[feature 2]**, and **[feature 3]**. These results **[agree / partly agree / differ]** from the Lab 2 EDA because **[mention the actual EDA pattern]**. Feature importance indicates how the tree used a feature for prediction; it does not by itself prove that the feature causes churn.


## 10. Interpretation

The Decision Tree experiment demonstrates the trade-off between model complexity and generalization. An unrestricted tree can fit the training data very closely, while limiting `max_depth` can reduce unnecessary complexity. The final choice should therefore consider test-set performance and the gap between training and test performance rather than training accuracy alone.

## 11. Conclusion

In [ ]:
# Task 21: Automatically prepare the main numbers for the conclusion.
chosen_train_accuracy = accuracy_score(y_train, chosen_model.predict(X_train))
chosen_test_accuracy = accuracy_score(y_test, chosen_model.predict(X_test))

print(f"Chosen model: Decision Tree with max_depth={selected_depth}")
print(f"Training accuracy: {chosen_train_accuracy:.4f}")
print(f"Test accuracy: {chosen_test_accuracy:.4f}")
print(f"Test precision: {chosen_metrics['Precision']:.4f}")
print(f"Test recall: {chosen_metrics['Recall']:.4f}")
print(f"Test F1-score: {chosen_metrics['F1-score']:.4f}")


### Task 21 — Conclusion draft

This lab used a Decision Tree classifier to predict customer churn from the cleaned Telco Customer Churn dataset. The baseline tree was compared with depth-controlled trees to investigate the effect of model complexity and overfitting. The selected model was chosen using test-set evidence rather than training accuracy alone, and its performance was evaluated using accuracy, precision, recall, F1-score, and a confusion matrix. The main limitation is that performance depends on the available features and the train/test split, and feature importance does not establish causation. Class imbalance can also affect how well the model detects the minority churn class.

### Task 22 — Next Steps

If more time were available, I would compare the Decision Tree with another suitable classification algorithm, investigate additional feature engineering, and experiment with methods for handling class imbalance. I would also consider cross-validation and threshold analysis in a later experiment so that model performance could be assessed more robustly.

## Task 23 — ID3 From Scratch: Play Badminton

In [ ]:
# No built-in Decision Tree library is used in this section.
# The implementation below calculates entropy and information gain
# and recursively builds an ID3 tree for the classic Play Badminton dataset.

play_data = pd.DataFrame({
    "Outlook": ["Sunny","Sunny","Overcast","Rain","Rain","Rain","Overcast","Sunny","Sunny","Rain","Sunny","Overcast","Overcast","Rain"],
    "Temperature": ["Hot","Hot","Hot","Mild","Cool","Cool","Cool","Mild","Cool","Mild","Mild","Mild","Hot","Mild"],
    "Humidity": ["High","High","High","High","Normal","Normal","Normal","High","Normal","Normal","Normal","High","Normal","High"],
    "Wind": ["Weak","Strong","Weak","Weak","Weak","Strong","Strong","Weak","Weak","Weak","Strong","Strong","Weak","Strong"],
    "Play": ["No","No","Yes","Yes","Yes","No","Yes","No","Yes","Yes","Yes","Yes","Yes","No"]
})

play_data


In [ ]:
from math import log2

def entropy(values):
    counts = pd.Series(values).value_counts()
    total = len(values)
    result = 0.0
    for count in counts:
        p = count / total
        result -= p * log2(p)
    return result

def information_gain(data, attribute, target):
    parent_entropy = entropy(data[target])
    weighted_entropy = 0.0

    for value, subset in data.groupby(attribute):
        weighted_entropy += (len(subset) / len(data)) * entropy(subset[target])

    return parent_entropy - weighted_entropy

def print_gain_values(data, target, indent=""):
    attributes = [c for c in data.columns if c != target]
    print(indent + f"Node: {len(data)} rows, entropy={entropy(data[target]):.4f}")
    for attr in attributes:
        print(indent + f"  Gain({attr}) = {information_gain(data, attr, target):.4f}")

def id3(data, target, indent=""):
    # Print gain values at every split.
    print_gain_values(data, target, indent)

    if len(data[target].unique()) == 1:
        return data[target].iloc[0]

    attributes = [c for c in data.columns if c != target]

    if not attributes:
        return data[target].mode()[0]

    gains = {attr: information_gain(data, attr, target) for attr in attributes}
    best_attribute = max(gains, key=gains.get)

    print(indent + f"-> Split on: {best_attribute}\n")

    tree = {best_attribute: {}}

    for value in data[best_attribute].unique():
        subset = data[data[best_attribute] == value].drop(columns=[best_attribute])

        if subset.empty:
            tree[best_attribute][value] = data[target].mode()[0]
        else:
            tree[best_attribute][value] = id3(subset, target, indent + "    ")

    return tree

play_tree = id3(play_data, "Play")
print("\nFinal ID3 tree:")
print(play_tree)


**Interpretation:** The implementation calculates entropy and information gain manually and chooses the attribute with the highest information gain at each split. The gain values printed at every recursive split provide the evidence for each ID3 decision.

## Submission Checklist

- [x] Student-info block included
- [x] Problem Definition
- [x] Dataset Verification
- [x] Feature/Target Preparation
- [x] Train/Test Split
- [x] Baseline Decision Tree
- [x] Model Evaluation
- [x] Overfitting Experiment
- [x] Model Comparison
- [x] Feature Importance
- [x] Interpretation
- [x] Conclusion
- [x] ID3 from scratch
- [x] Written interpretations included after tables/plots

**Before submission:** upload `clean_churn.csv` to Colab, run all cells from top to bottom, verify the actual numerical outputs, and replace the Task 20 EDA placeholder with the specific findings from your Lab 2 notebook.
